# Paso 3 (automático) — Detección por modalidad en Colab, SIN Roboflow ni anotación

Este cuaderno entrena un **YOLO por modalidad de entrada** (VIS · IR · Torre Top-Hat *anterior* · **Óptimo Multiescala**) usando los **20 pares TNO de tu propio repo** y **auto-generando las etiquetas** con un YOLO preentrenado (pseudo-etiquetas). No requiere Roboflow, ni API key, ni anotar a mano.

**Cómo usar:** Subí este `.ipynb` a [Colab](https://colab.research.google.com) → *Entorno de ejecución → Cambiar tipo → GPU* → *Ejecutar todo*. (~10–20 min con GPU T4.)


## 1. Clonar tu repositorio e instalar


In [ ]:
!git clone https://github.com/YanBajac/tesis_mciencias_datos.git
%cd tesis_mciencias_datos
!pip -q install ultralytics
import torch; print('GPU disponible:', torch.cuda.is_available())


## 2. Entrenar YOLO por modalidad (pseudo-etiquetas)
Genera pseudo-etiquetas con un YOLO preentrenado sobre VIS, construye los datasets de las 4 modalidades (mismas cajas, imágenes registradas) y entrena un YOLO por cada una. Reporta mAP@0.5, mAP@0.5:0.95, P, R, F1.

Empezá con `--epochs 30`; si querés una prueba rápida primero, usá `--epochs 10`.


In [ ]:
!python experiments/detection/train_local_pseudolabels.py --epochs 30 --device 0


## 3. Resultados


In [ ]:
import pandas as pd
df = pd.read_csv('experiments/results/metrics_reports/detection_training_metrics.csv')
display(df)
import matplotlib.pyplot as plt
df.set_index('modalidad')[['mAP50','mAP50_95','precision','recall','F1']].plot.bar(figsize=(10,5))
plt.title('Detección YOLO por modalidad (pseudo-etiquetas)'); plt.ylim(0,1); plt.tight_layout(); plt.show()


## 4. Curvas y descarga
Las curvas de pérdida por modalidad están en `experiments/results/detection_train/runs/<modalidad>/`. Descargá el CSV y pasámelo para integrar la **Parte B (aplicación)** de la tesis:


In [ ]:
from google.colab import files
files.download('experiments/results/metrics_reports/detection_training_metrics.csv')


> **Nota metodológica:** las etiquetas son pseudo-etiquetas (lo que un YOLO preentrenado detecta en VIS), por lo que la comparación es **relativa entre modalidades** —ideal para aislar el efecto de la fusión—, no un mAP absoluto frente a anotación humana. Para mAP absoluto, usá el notebook `04` con un dataset etiquetado en Roboflow.
